# Hobby Corner RMS → Lightspeed Product Migration
**Analysis / dry-run only. This notebook does not write to Lightspeed.**

RMS on-hand quantity is used only as a migration-selection signal. Opening inventory is always 0 here; the physical inventory at cutover establishes actual Lightspeed stock.

In [ ]:
from google.colab import auth
auth.authenticate_user()

!pip -q install gspread pandas openpyxl

import os
import subprocess
import sys

NOTEBOOK_VERSION = '2026-10-08-v9'
print('Notebook version:', NOTEBOOK_VERSION)

repo_dir = '/content/hobby-corner'
if os.path.exists(os.path.join(repo_dir, '.git')):
    subprocess.run(['git', '-C', repo_dir, 'fetch', 'origin', 'product-migration-colab'], check=True)
    subprocess.run(['git', '-C', repo_dir, 'checkout', 'product-migration-colab'], check=True)
    subprocess.run(['git', '-C', repo_dir, 'reset', '--hard', 'origin/product-migration-colab'], check=True)
else:
    subprocess.run([
        'git', 'clone', '-b', 'product-migration-colab',
        'https://github.com/Tawgod/Hobby-Corner-Discord-Bot.git', repo_dir
    ], check=True)

sys.path.insert(0, '/content/hobby-corner/migration/product_migration')


> Expected version: **2026-10-08-v9**. This version validates brand, UPC, supplier code, weight, and picture source in the test preview.

In [ ]:
import google.auth
import gspread
import pandas as pd
from datetime import datetime, timezone
import importlib
import analysis
import test_harness

importlib.reload(analysis)
importlib.reload(test_harness)
from analysis import (
    find_category_mapping_sheet,
    find_supplier_mapping_sheet,
    build_category_map,
    build_supplier_map,
    analyze_candidates,
    split_outputs,
    summary,
)
from test_harness import (
    build_category_uuid_map,
    build_brand_uuid_map,
    build_supplier_uuid_map,
    select_test_batch,
    build_test_preview,
)

creds, _ = google.auth.default()
gc = gspread.authorize(creds)


## Source spreadsheets
The RMS candidate sheet is the export from RMS. The LS importer is authoritative for supplier/category destination mappings.

In [ ]:
RMS_CANDIDATE_SHEET = 'https://docs.google.com/spreadsheets/d/1_M4b0iD8EG3pn-cuR_uGZ3KoXL0bA1A_OpusCJboLSQ/edit'
LS_IMPORTER_SHEET = 'https://docs.google.com/spreadsheets/d/1-F1LpskwwkfiTJzQjpVmZHjOo7iOnCzcFw4CmOiN5kI/edit'

def workbook_frames(url):
    book = gc.open_by_url(url)
    frames = {}
    for ws in book.worksheets():
        values = ws.get_all_values()
        if not values:
            frames[ws.title] = pd.DataFrame()
            continue
        header_idx = next((i for i, r in enumerate(values) if any(str(x).strip() for x in r)), None)
        if header_idx is None:
            frames[ws.title] = pd.DataFrame()
            continue
        headers = values[header_idx]
        data = values[header_idx + 1:]
        frames[ws.title] = pd.DataFrame(data, columns=headers)
    return frames

def load_candidate_upload():
    from google.colab import files
    import io
    print('The RMS candidate Google Sheet is empty. Upload the RMS candidate CSV or XLSX now.')
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('No candidate file was uploaded.')
    filename, raw = next(iter(uploaded.items()))
    lower = filename.lower()
    if lower.endswith('.csv'):
        df = pd.read_csv(io.BytesIO(raw), dtype=str, keep_default_na=False)
    elif lower.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(io.BytesIO(raw), dtype=str, keep_default_na=False)
    else:
        raise ValueError('Please upload a CSV or XLSX candidate export.')
    return {f'UPLOAD: {filename}': df}

rms_tabs = workbook_frames(RMS_CANDIDATE_SHEET)
if not any(not df.empty and len(df.columns) > 0 for df in rms_tabs.values()):
    rms_tabs = load_candidate_upload()

ls_tabs = workbook_frames(LS_IMPORTER_SHEET)
print('RMS source tabs:', list(rms_tabs))
print('LS importer tabs:', list(ls_tabs))


In [ ]:
# Defensive check: Colab can retain variables from an older notebook run.
if not any(not df.empty and len(df.columns) > 0 for df in rms_tabs.values()):
    print('RMS source currently contains no product data; switching to file upload.')
    rms_tabs = load_candidate_upload()

# The RMS export has evolved over time, so normalize common header variants to canonical names.
ALIASES = {
    'RMS_ItemID': ['RMS_ItemID', 'ItemID', 'ID'],
    'RMS_SKU': ['RMS_SKU', 'ItemLookupCode', 'SKU', 'Item Lookup Code'],
    'RMS_Description': ['RMS_Description', 'Description', 'ItemDescription'],
    'RMS_Department': ['RMS_Department', 'RMS_DepartmentName', 'Department', 'DepartmentName', 'Department Name'],
    'RMS_Category': ['RMS_Category', 'RMS_CategoryName', 'Category', 'CategoryName', 'Category Name'],
    'RMS_Supplier': ['RMS_Supplier', 'RMS_SupplierName', 'Supplier', 'SupplierName', 'Supplier Name'],
    'RMS_Quantity': ['RMS_Quantity', 'Quantity', 'On Hand', 'OnHand'],
    'RMS_ReorderPoint': ['RMS_ReorderPoint', 'ReorderPoint', 'reorder_point'],
    'RMS_RestockLevel': ['RMS_RestockLevel', 'RestockLevel', 'restock_level'],
    'LastSold': ['LastSold', 'last_sold'],
    'LastReceived': ['LastReceived', 'last_received'],
    'Inactive': ['Inactive'],
    'DoNotOrder': ['DoNotOrder', 'Do Not Order'],
}

def _header_key(value):
    return ''.join(ch.lower() for ch in str(value).strip() if ch.isalnum())

def normalize_candidate_headers(df):
    lookup = {_header_key(c): c for c in df.columns}
    renames = {}
    found = {}
    for canonical, aliases in ALIASES.items():
        actual = next((lookup.get(_header_key(a)) for a in aliases if _header_key(a) in lookup), None)
        if actual is not None:
            renames[actual] = canonical
            found[canonical] = actual
    return df.rename(columns=renames).copy(), found

# Score each RMS tab and choose the one that looks most like the product candidate export.
scored = []
for name, df in rms_tabs.items():
    normalized, found = normalize_candidate_headers(df)
    core = sum(k in found for k in ['RMS_ItemID', 'RMS_SKU', 'RMS_Description'])
    context = sum(k in found for k in ['RMS_Department', 'RMS_Category', 'RMS_Supplier', 'RMS_Quantity'])
    scored.append((core * 10 + context, name, normalized, found))

scored.sort(key=lambda x: x[0], reverse=True)
if not scored or scored[0][0] < 30:
    print('Could not identify the RMS product candidate tab automatically.')
    for tab_name, df in rms_tabs.items():
        print(f'\n{tab_name}:')
        print(list(df.columns))
    raise ValueError('No RMS tab contained recognizable item ID, SKU, and description columns.')

_, candidate_name, candidates, found_headers = scored[0]
print('Candidate tab:', candidate_name)
print('Recognized RMS headers:', found_headers)
print('Canonical candidate columns:', list(candidates.columns))

missing_context = [c for c in ['RMS_Department', 'RMS_Category'] if c not in candidates.columns]
if missing_context:
    raise ValueError(f'Candidate export is missing required category context after normalization: {missing_context}')

category_tab, category_df = find_category_mapping_sheet(ls_tabs)
supplier_tab, supplier_df, supplier_source_col, supplier_target_col = find_supplier_mapping_sheet(ls_tabs)

print('Category mapping tab:', category_tab)
print('Supplier mapping tab:', supplier_tab, f'({supplier_source_col} -> {supplier_target_col})')


In [ ]:
category_map = build_category_map(category_df)
supplier_map = build_supplier_map(supplier_df, supplier_source_col, supplier_target_col)
result = analyze_candidates(candidates, category_map, supplier_map)
groups = split_outputs(result)

display(summary(result))

migration_candidates = result[result['migration_status'].isin(['READY', 'REVIEW'])].copy()
all_unmapped_categories = int((~result['Category_Mapped']).sum())
all_unmapped_suppliers = int((~result['Supplier_Mapped']).sum())
candidate_unmapped_categories = int((~migration_candidates['Category_Mapped']).sum())
candidate_unmapped_suppliers = int((~migration_candidates['Supplier_Mapped']).sum())

print('All-catalog unmapped categories:', all_unmapped_categories)
print('All-catalog unmapped suppliers:', all_unmapped_suppliers)
print('Migration-candidate unmapped categories:', candidate_unmapped_categories)
print('Migration-candidate unmapped suppliers:', candidate_unmapped_suppliers)
print('Opening inventory imported:', result['Opening_Inventory_To_Import'].sum())

# Rank only migration candidates, because EXCLUDE rows do not need mapping cleanup.
unmapped_category_summary = (
    migration_candidates[~migration_candidates['Category_Mapped']]
    .groupby(['RMS_Department', 'RMS_Category'], dropna=False)
    .size()
    .reset_index(name='Candidate Products')
    .sort_values('Candidate Products', ascending=False)
)

unmapped_supplier_summary = (
    migration_candidates[~migration_candidates['Supplier_Mapped']]
    .groupby(['RMS_Supplier'], dropna=False)
    .size()
    .reset_index(name='Candidate Products')
    .sort_values('Candidate Products', ascending=False)
)

print('\nTop unmapped categories among migration candidates:')
display(unmapped_category_summary.head(30))
print('\nTop unmapped suppliers among migration candidates:')
display(unmapped_supplier_summary.head(30))


## Write analysis results back to LS: importer
This updates only reporting tabs in the Google Sheet. It still does not write anything to Lightspeed.

In [ ]:
ls_book = gc.open_by_url(LS_IMPORTER_SHEET)

def get_or_create_tab(book, title, rows=1000, cols=30):
    try:
        return book.worksheet(title)
    except gspread.WorksheetNotFound:
        return book.add_worksheet(title=title, rows=rows, cols=cols)

run_ws = get_or_create_tab(ls_book, 'Migration Run Results', rows=500, cols=12)
review_ws = get_or_create_tab(ls_book, 'Migration Review', rows=max(len(groups['REVIEW']) + 10, 1000), cols=30)
cat_gap_ws = get_or_create_tab(ls_book, 'Migration Unmapped Categories', rows=max(len(unmapped_category_summary) + 10, 1000), cols=10)
supplier_gap_ws = get_or_create_tab(ls_book, 'Migration Unmapped Suppliers', rows=max(len(unmapped_supplier_summary) + 10, 1000), cols=10)

run_time = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
summary_row = [
    run_time,
    len(result),
    len(groups['READY']),
    len(groups['REVIEW']),
    len(groups['EXCLUDE']),
    all_unmapped_categories,
    all_unmapped_suppliers,
    candidate_unmapped_categories,
    candidate_unmapped_suppliers,
    float(result['Opening_Inventory_To_Import'].sum()),
    candidate_name,
    category_tab,
    supplier_tab,
]
headers = ['Run Time', 'Total', 'READY', 'REVIEW', 'EXCLUDE', 'All Unmapped Categories', 'All Unmapped Suppliers', 'Candidate Unmapped Categories', 'Candidate Unmapped Suppliers', 'Opening Inventory Imported', 'Candidate Tab', 'Category Mapping Tab', 'Supplier Mapping Tab']

if not run_ws.get_all_values():
    run_ws.append_row(headers, value_input_option='RAW')
run_ws.append_row(summary_row, value_input_option='RAW')

review_cols = [c for c in [
    'RMS_ItemID','RMS_SKU','RMS_Description','RMS_Department','RMS_Category','RMS_Supplier',
    'RMS_Quantity','LastSold','LastReceived','LS_Category_Path','LS_Supplier',
    'proposed_reorder_point','proposed_restock_level','review_reason'
] if c in result.columns]

review_out = groups['REVIEW'][review_cols].copy().fillna('')
review_ws.clear()
review_ws.update([review_out.columns.tolist()] + review_out.astype(str).values.tolist(), 'A1')

print('Saved run summary to: Migration Run Results')
print('Saved review queue to: Migration Review')

# Write ranked mapping gaps for review.
cat_gap_ws.clear()
cat_gap_out = unmapped_category_summary.fillna('')
cat_gap_ws.update([cat_gap_out.columns.tolist()] + cat_gap_out.astype(str).values.tolist(), 'A1')
supplier_gap_ws.clear()
supplier_gap_out = unmapped_supplier_summary.fillna('')
supplier_gap_ws.update([supplier_gap_out.columns.tolist()] + supplier_gap_out.astype(str).values.tolist(), 'A1')
print('Saved category mapping gaps to: Migration Unmapped Categories')
print('Saved supplier mapping gaps to: Migration Unmapped Suppliers')


## Controlled Lightspeed product test preview
This stage selects a varied fully mapped batch and generates the exact 2026-10 product-family payloads. **It still does not call Lightspeed.**

In [ ]:
# Resolve current Lightspeed category, brand, and supplier UUID maps from LS: importer.
cat_id_df = ls_tabs.get('Cat ID')
brand_id_df = ls_tabs.get('Brand ID Map')
supplier_id_df = ls_tabs.get('Supplier ID Map')
if cat_id_df is None or brand_id_df is None or supplier_id_df is None:
    raise ValueError("LS: importer must contain 'Cat ID', 'Brand ID Map', and 'Supplier ID Map' tabs.")

category_uuid_map = build_category_uuid_map(cat_id_df)
brand_uuid_map = build_brand_uuid_map(brand_id_df)
supplier_uuid_map = build_supplier_uuid_map(supplier_id_df)

# RMS is a US store database and Hobby Corner has historically treated the
# item Weight field as pounds. Change this one setting if source validation
# shows a different unit. Lightspeed accepts CT/G/OZ/LB/KG.
DEFAULT_RMS_WEIGHT_UNIT = 'LB'

test_batch = select_test_batch(result, size=8)
test_preview, test_payloads = build_test_preview(
    test_batch,
    category_uuid_map,
    supplier_uuid_map,
    brand_uuid_map=brand_uuid_map,
    default_weight_unit=DEFAULT_RMS_WEIGHT_UNIT,
)

display(test_preview.drop(columns=['Payload JSON']))
print('Test run ID:', test_preview.iloc[0]['Run ID'])
print('Products selected:', len(test_preview))
print('Opening inventory total:', test_preview['Opening Inventory'].sum())
print('Lightspeed write enabled:', bool(test_preview['Write Enabled'].any()))
print('Rows missing rich fields:', int(test_preview['Missing Rich Fields'].astype(bool).sum()))


In [ ]:
# Write the preview into LS: importer so it can be reviewed outside Colab.
ls_book = gc.open_by_url(LS_IMPORTER_SHEET)
try:
    test_ws = ls_book.worksheet('Migration Test Preview')
except gspread.WorksheetNotFound:
    test_ws = ls_book.add_worksheet(title='Migration Test Preview', rows=500, cols=20)

test_ws.clear()
preview_out = test_preview.fillna('')
test_ws.update([preview_out.columns.tolist()] + preview_out.astype(str).values.tolist(), 'A1')
print('Saved test preview to: Migration Test Preview')


### Payload inspection
Inspect these JSON payloads before enabling writes. The payloads contain no `outlet_inventories` field.

In [ ]:
import json
for i, payload in enumerate(test_payloads, start=1):
    print(f'\n--- Test product {i} ---')
    print(json.dumps(payload, indent=2))
